# GKE Capacity and Rightsizing Notebook

Goal: compare Kubernetes requests with p95 usage and identify over-provisioned or under-provisioned workloads.

In [ ]:
from pathlib import Path
import pandas as pd

df = pd.read_csv(Path('../data/k8s_resource_usage.csv'))
df['cpu_headroom_ratio'] = df['cpu_request_millicores'] / df['cpu_usage_p95_millicores']
df['memory_headroom_ratio'] = df['memory_request_mib'] / df['memory_usage_p95_mib']
df

In [ ]:
def recommendation(row):
    if row['cpu_headroom_ratio'] < 0.8 or row['memory_headroom_ratio'] < 0.8:
        return 'increase requests or scale out'
    if row['cpu_headroom_ratio'] > 2.0 and row['memory_headroom_ratio'] > 1.5:
        return 'reduce requests'
    return 'keep current sizing'

df['recommendation'] = df.apply(recommendation, axis=1)
df[['namespace', 'workload', 'cpu_headroom_ratio', 'memory_headroom_ratio', 'recommendation']]

In [ ]:
namespace_summary = df.groupby('namespace', as_index=False).agg(
    total_cpu_request_m=('cpu_request_millicores', 'sum'),
    total_memory_request_mib=('memory_request_mib', 'sum'),
    workloads=('workload', 'count')
)
namespace_summary

## Interview Talking Points

- Requests affect scheduling and cluster capacity.
- Limits affect throttling and OOM behavior.
- Rightsizing reduces cost without hurting reliability.